# Chapter 16 — Exceptions, logging, configuration, and user-safe diagnostics

**Book role:** Application design  
**Suggested book space:** about 8 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Distinguish expected input errors, I/O errors, and programming bugs.
- Report actionable user-facing errors while preserving useful logs.
- Configure settings without hard-coded machine paths.
- Load environment configuration safely.

## 1. Handle errors at the boundary where they become meaningful

A user typing an invalid date is an expected validation error; a workbook locked by another process is an I/O error; a missing required column is a data-schema problem; an unexpected `AttributeError` may indicate a programming defect. These cases deserve different treatment. Catch specific exceptions close to the operation that can recover from them, and avoid blanket `except Exception: pass` blocks.

The GUI translates a failure into a useful message. It should preserve work where possible and explain a next step—for example, “Column `order_date` is missing; choose a file with the required sales schema.” Diagnostic logs can include the filename, operation, exception stack, and correlation context but should not expose credentials or unnecessary personal data.

### Worked example 1: Configure a file logger

The handler rotates logs at a size limit. Production code should select a platform-appropriate application-data directory and document retention and privacy settings.

In [ ]:
import logging
from logging.handlers import RotatingFileHandler
from pathlib import Path

log_dir = Path.home() / ".tkinter_analytics" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / "application.log"

logger = logging.getLogger("analytics_app")
logger.setLevel(logging.INFO)
if not logger.handlers:  # Avoid duplicate handlers during repeated notebook runs.
    handler = RotatingFileHandler(log_path, maxBytes=1_000_000, backupCount=3, encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s %(message)s"))
    logger.addHandler(handler)
logger.info("Application logging initialized")
print("Log path:", log_path)

## 2. Logging and rotating diagnostics

Use the `logging` module instead of scattering `print()` calls through production code. DEBUG records help development; INFO records describe major state transitions; WARNING records signal unusual but recoverable events; ERROR records describe operations that failed. Configure logging once in the application entry point and use `logging.getLogger(__name__)` in modules.

A file handler allows bug reports to include a log, but logs need a retention and privacy policy. Do not log connection strings, authentication tokens, full private records, or spreadsheet contents by default. Limit size or rotate logs, make the log location discoverable, and give the user a way to open the log folder when support is needed.

### Worked example 2: Parse an environment variable with a safe default

The host is not secret in many setups, but passwords and complete URIs should not be printed. Validate configuration according to the application’s allowed operating range.

In [ ]:
import os

raw_page_size = os.getenv("APP_PAGE_SIZE", "200")
try:
    page_size = int(raw_page_size)
    if not 20 <= page_size <= 2000:
        raise ValueError("outside allowed range")
except ValueError:
    page_size = 200
    print("Invalid APP_PAGE_SIZE; using safe default 200")

# Do not print database credentials or entire connection URIs.
mysql_host = os.getenv("MYSQL_HOST", "localhost")
print("MySQL host configured:", bool(mysql_host))

## 3. Configuration versus secrets

Ordinary settings include the default data directory, preferred theme, page size, and last-used filters. Secrets include database passwords, API tokens, and private certificates. Keep secrets in environment variables or a secret manager rather than source code or shared notebooks. A `.env` file can be convenient for local development, but keep it out of version control and use a non-secret `.env.example` with placeholder names only.

Validate configuration at startup. A missing optional MongoDB URI should disable that connector’s actions rather than make CSV features unusable. A malformed page-size setting should fall back or show an explicit configuration error. Do not silently choose a production database because a configuration variable was missing.

## Guided lab: Lab: user-safe failure handling

Define error categories for invalid input, missing schema, file access failure, database connectivity failure, query failure, and unexpected internal error. For each, write a user-facing message and a corresponding log event. Ensure logs do not contain credentials, tokens, or full rows. Test an unreadable file and a deliberately missing required column.

## Review questions and practice

1. Why should expected validation errors be treated differently from programming defects?
2. Which logging levels would you use for recoverable anomalies and failed operations?
3. Why should a database URI not be printed to logs?
4. What belongs in `.env.example` rather than `.env`?

## Further reading

- Logging HOWTO: https://docs.python.org/3/howto/logging.html
- Environment variables: https://docs.python.org/3/library/os.html#os.environ

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.